# Q. Multiclass Logistic Regression: Automatic Classification of Emails

### Maximum Entropy Derivation and Numerical Solution

## 1. Setting Up the Problem

The upstream AI Feature Extractor Agent outputs a discrete binary vector $\mathbf{x}=(x_1,x_2,x_3)^T$, and the routing target $Y\in\{0,1\}$ where $Y=0$ is **Finance** and $Y=1$ is **Technical**. (The problem statement mentions four destinations but only specifies and constrains two, $Y=0$ and $Y=1$; the derivation below is carried out for the *general multiclass* case and then specialized to the binary case actually supported by the given data, since the same softmax/logistic machinery covers both.)

**Known marginal distribution of $\mathbf{x}$** (from historical logs):

| Scenario | $\mathbf{x}=(x_1,x_2,x_3)$ | $P(\mathbf{x})$ |
|---|---|---|
| A (Pure Financial) | $(1,0,0)$ | $0.40$ |
| B (Pure Access) | $(0,1,0)$ | $0.25$ |
| C (Pure Technical) | $(0,0,1)$ | $0.05$ |
| D (Access + Technical) | $(0,1,1)$ | $0.15$ |
| E (Financial + Access) | $(1,1,0)$ | $0.05$ |

**Moment constraints from manual audits:**

$$\mathbb{E}[x_1\,\mathbb{I}(Y=1)] = 0.05, \qquad \mathbb{E}[x_2\,\mathbb{I}(Y=1)] = 0.15, \qquad \mathbb{E}[x_3\,\mathbb{I}(Y=1)] = 0.16$$

We want the **least-biased** conditional law $P(Y=1\mid \mathbf{x})$ consistent with the known marginal $P(\mathbf{x})$ and these three moment constraints.

## 2. Maximum Entropy Derivation

Since $P(\mathbf{x})$ is already fixed by the historical logs, maximizing the entropy of the joint law $P(\mathbf{x},y)$ subject to the constraints is equivalent to maximizing the **conditional entropy**

$$H(Y\mid X) = -\sum_{\mathbf{x}} P(\mathbf{x}) \sum_{y} P(y\mid \mathbf{x}) \ln P(y\mid \mathbf{x})$$

subject to, for every $\mathbf{x}$, the normalization $\sum_y P(y\mid \mathbf{x}) = 1$, and the moment constraints

$$\sum_{\mathbf{x}} P(\mathbf{x})\, x_i\, P(Y=1\mid \mathbf{x}) = c_i, \qquad i=1,2,3.$$

**Lagrangian.** Introduce a multiplier $\alpha_{\mathbf{x}}$ for each normalization constraint and $\lambda_i$ for each moment constraint:

$$\mathcal{L} = -\sum_{\mathbf{x}} P(\mathbf{x})\sum_y P(y\mid \mathbf{x})\ln P(y\mid \mathbf{x}) + \sum_{\mathbf{x}} \alpha_{\mathbf{x}}\Big[\sum_y P(y\mid \mathbf{x}) - 1\Big] + \sum_{i=1}^{3} \lambda_i \Big[\sum_{\mathbf{x}} P(\mathbf{x})\,x_i\, P(1\mid \mathbf{x}) - c_i\Big]$$

Differentiating with respect to $P(1\mid\mathbf{x})$ and $P(0\mid\mathbf{x})$ and setting each to zero:

$$\frac{\partial \mathcal{L}}{\partial P(1\mid\mathbf{x})} = -P(\mathbf{x})\big[\ln P(1\mid\mathbf{x})+1\big] + \alpha_{\mathbf{x}} + P(\mathbf{x})\sum_i \lambda_i x_i = 0$$
$$\frac{\partial \mathcal{L}}{\partial P(0\mid\mathbf{x})} = -P(\mathbf{x})\big[\ln P(0\mid\mathbf{x})+1\big] + \alpha_{\mathbf{x}} = 0$$

Subtracting the two equations eliminates $\alpha_{\mathbf{x}}$ and $P(\mathbf{x})$ cleanly:

$$\ln\frac{P(1\mid \mathbf{x})}{P(0\mid \mathbf{x})} = \sum_{i=1}^3 \lambda_i x_i = \boldsymbol{\lambda}^T\mathbf{x}$$

Combined with $P(1\mid\mathbf{x}) + P(0\mid \mathbf{x}) = 1$, this gives the **logistic (sigmoid) form**:

$$\boxed{\,P(Y=1\mid \mathbf{x}) = \sigma(\boldsymbol{\lambda}^T\mathbf{x}) = \frac{e^{\boldsymbol{\lambda}^T\mathbf{x}}}{1+e^{\boldsymbol{\lambda}^T\mathbf{x}}}\,}, \qquad P(Y=0\mid \mathbf{x}) = \frac{1}{1+e^{\boldsymbol{\lambda}^T\mathbf{x}}}$$

**This is precisely logistic regression** — the maximum entropy distribution consistent with feature/label moment matching constraints is the (multiclass, softmax-generalized) logistic model. For $K$ classes the same argument yields the softmax form $P(Y=k\mid\mathbf{x}) \propto \exp(\boldsymbol{\lambda}_k^T\mathbf{x})$; here $K=2$ reduces it to a single weight vector and the sigmoid.

The multipliers $\boldsymbol{\lambda}=(\lambda_1,\lambda_2,\lambda_3)$ are then pinned down by requiring the model to reproduce exactly the three observed moments — this is done numerically below.

## 3. The Estimating Equations

For each scenario the linear score $z=\boldsymbol{\lambda}^T\mathbf{x}$ is:

| Scenario | $\mathbf x$ | $z=\boldsymbol\lambda^T\mathbf x$ | $P(Y{=}1\mid \mathbf x)$ |
|---|---|---|---|
| A | $(1,0,0)$ | $\lambda_1$ | $\sigma(\lambda_1)$ |
| B | $(0,1,0)$ | $\lambda_2$ | $\sigma(\lambda_2)$ |
| C | $(0,0,1)$ | $\lambda_3$ | $\sigma(\lambda_3)$ |
| D | $(0,1,1)$ | $\lambda_2+\lambda_3$ | $\sigma(\lambda_2+\lambda_3)$ |
| E | $(1,1,0)$ | $\lambda_1+\lambda_2$ | $\sigma(\lambda_1+\lambda_2)$ |

Substituting into the three moment constraints gives a $3\times3$ nonlinear system in $\boldsymbol\lambda$:

$$0.40\,\sigma(\lambda_1) + 0.05\,\sigma(\lambda_1+\lambda_2) = 0.05$$
$$0.25\,\sigma(\lambda_2) + 0.15\,\sigma(\lambda_2+\lambda_3) + 0.05\,\sigma(\lambda_1+\lambda_2) = 0.15$$
$$0.05\,\sigma(\lambda_3) + 0.15\,\sigma(\lambda_2+\lambda_3) = 0.16$$

This is solved numerically below (a closed form is not available because the same $\lambda$'s appear in multiple overlapping equations).

In [1]:
import numpy as np
from scipy.optimize import fsolve

# Marginal probabilities of each observed x-scenario
p = {'A': 0.40, 'B': 0.25, 'C': 0.05, 'D': 0.15, 'E': 0.05}

# Feature vectors x = (x1, x2, x3) for each scenario
x = {
    'A': np.array([1, 0, 0]),
    'B': np.array([0, 1, 0]),
    'C': np.array([0, 0, 1]),
    'D': np.array([0, 1, 1]),
    'E': np.array([1, 1, 0]),
}

# Target moments E[x_i * I(Y=1)]
c = np.array([0.05, 0.15, 0.16])

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def P1(lam, scenario):
    """Maximum-entropy P(Y=1 | x) for a given scenario and lambda vector."""
    return sigmoid(lam @ x[scenario])

def equations(lam):
    eq1 = p['A']*P1(lam, 'A') + p['E']*P1(lam, 'E') - c[0]
    eq2 = p['B']*P1(lam, 'B') + p['D']*P1(lam, 'D') + p['E']*P1(lam, 'E') - c[1]
    eq3 = p['C']*P1(lam, 'C') + p['D']*P1(lam, 'D') - c[2]
    return [eq1, eq2, eq3]

lam0 = np.zeros(3)
lam_star, info, ier, msg = fsolve(equations, lam0, full_output=True)

print("Convergence:", msg)
print("lambda_1, lambda_2, lambda_3 =", np.round(lam_star, 4))
print("Residuals of constraint equations:", np.round(equations(lam_star), 8))

Convergence: The solution converged.
lambda_1, lambda_2, lambda_3 = [-1.9725 -1.7762  2.8804]
Residuals of constraint equations: [ 0. -0.  0.]


In [2]:
print(f"{'Scenario':<10}{'x':<12}{'P(Y=1|x)':<14}{'P(Y=0|x)':<14}")
print("-" * 50)
for s in ['A', 'B', 'C', 'D', 'E']:
    p1 = P1(lam_star, s)
    print(f"{s:<10}{str(tuple(x[s])):<12}{p1:<14.4f}{1-p1:<14.4f}")

print("\nVerification of the three moment constraints:")
check = {
    'E[x1*I(Y=1)]': p['A']*P1(lam_star,'A') + p['E']*P1(lam_star,'E'),
    'E[x2*I(Y=1)]': p['B']*P1(lam_star,'B') + p['D']*P1(lam_star,'D') + p['E']*P1(lam_star,'E'),
    'E[x3*I(Y=1)]': p['C']*P1(lam_star,'C') + p['D']*P1(lam_star,'D'),
}
target = {'E[x1*I(Y=1)]': 0.05, 'E[x2*I(Y=1)]': 0.15, 'E[x3*I(Y=1)]': 0.16}
for k in check:
    print(f"{k}: computed = {check[k]:.5f}, target = {target[k]:.5f}")

Scenario  x           P(Y=1|x)      P(Y=0|x)      
--------------------------------------------------
A         (np.int64(1), np.int64(0), np.int64(0))0.1221        0.8779        
B         (np.int64(0), np.int64(1), np.int64(0))0.1448        0.8552        
C         (np.int64(0), np.int64(0), np.int64(1))0.9469        0.0531        
D         (np.int64(0), np.int64(1), np.int64(1))0.7510        0.2490        
E         (np.int64(1), np.int64(1), np.int64(0))0.0230        0.9770        

Verification of the three moment constraints:
E[x1*I(Y=1)]: computed = 0.05000, target = 0.05000
E[x2*I(Y=1)]: computed = 0.15000, target = 0.15000
E[x3*I(Y=1)]: computed = 0.16000, target = 0.16000


## 4. Results and Interpretation

Solving the estimating equations numerically gives

$$\lambda_1 \approx -1.972, \qquad \lambda_2 \approx -1.776, \qquad \lambda_3 \approx 2.880$$

so that the maximum-entropy classifier is

$$P(Y{=}1\mid \mathbf{x}) = \sigma\big(-1.972\,x_1 - 1.776\,x_2 + 2.880\,x_3\big)$$

**Sanity checks against the problem narrative:**

* $\lambda_1<0$: detecting *financial* context strongly pushes the email **away** from Technical routing — consistent with the narrative that financial keywords are "rarely routed to Technical support."
* $\lambda_2<0$ (and $|\lambda_2|<|\lambda_1|$): detecting an *account-access* context mildly discourages Technical routing on its own, but is easily overridden — consistent with the "splits down the middle depending on context" description, since $P(Y{=}1\mid \mathbf{x}_B)\approx 0.14$ alone but jumps to $\approx 0.75$ once a technical/bug signal is also present (scenario D).
* $\lambda_3>0$ and large: a *bug/malfunction* signal strongly pushes toward Technical routing, giving $P(Y{=}1\mid \mathbf{x}_C)\approx 0.95$ for a pure technical email, matching the "vast majority of the time" description and reproducing the stated $80\%$ conditional rate once averaged with scenario D through $P(\mathbf{x}_3{=}1) = P(C)+P(D) = 0.20$.

**Key conceptual takeaway:** this exercise shows that *logistic regression is not an arbitrary modeling choice* — it is the unique conditional distribution $P(Y\mid\mathbf{x})$ that has **maximum entropy** (i.e., is maximally non-committal / least biased) among all distributions consistent with the observed feature–label moments. The weights $\lambda_i$ found here play exactly the role of the logistic regression coefficients, and would be identical to what a maximum-likelihood logistic regression fit converges to on data generating these same empirical moments, since MaxEnt-with-moment-matching and MLE-of-the-logistic-model are dual optimization problems.